In [1]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

'64a5c402-05c4-4607-bbad-46a9c2aebd98'

In [2]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

agent=agent_local
# agent=agent_ssh
# agent=agent_slurm
agent.Deploy()

2025-03-13_00-39-38  | /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-03-13_00-39-38  | /home/tony
2025-03-13_00-39-38  | >>> mkdir -p $AGENT_HOME
2025-03-13_00-39-38  | >>> mkdir -p /home/tony/.globus
2025-03-13_00-39-38  | >>> mkdir -p /home/tony/.globusonline


2025-03-13_00-39-38 E| mkdir: missing operand
2025-03-13_00-39-38 E| Try 'mkdir --help' for more information.


2025-03-13_00-39-38  | >>> [ -e /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif ] || apptainer pull /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-13_00-39-39  | staged [msm_stub]
2025-03-13_00-39-39  | staged [msm]
2025-03-13_00-39-39  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-03-13_00-39-39  | including dev binds
2025-03-13_00-39-39  | 2025-03-13_00-39-39  | api call to [deploy_from_container] with [{}]
2025-03-13_00-39-39  | 2025-03-13_00-39-39  | deploying to [/ws]
2025-03-13_00-39-39  | 2025-03-13_00-39-39  | deploying relay server to [/ws/relay/msm_relay]
2025-03-13_00-39-39  | 2025-03-13_00-39-39  | deployment complete
2025-03-13_00-39-40  | staged [lib/agent.yml]
2025-03-13_00-39-40  | staged [lib/msm_bootstrap]
2025-03-13_00-39-40  | staged [lib/nextflow_config]
2025-03-13_00-39

In [3]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [4]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [5]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

dwfuH8Cz
pprodigal
diamond


In [6]:
# task.config = dict(
#     nextflow = dict(
#         preset = "slurm",
#     ),
# )

In [7]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-13_00-39-40  | connecting to deployed agent
2025-03-13_00-39-40  | starting relay service


 E| > 2025-03-13_00-39-41 E| relay server already running in [relay/connections]


  | > 2025-03-13_00-39-41  | connecting to relay as [xQLATWHmswnk]


2025-03-13_00-39-42 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
2025-03-13_00-39-42 W| clearing previously staged task


2025-03-13_00-39-42  | sending metadata for workflow [dwfuH8Cz]
2025-03-13_00-39-44  | staging
  | > including dev binds
  | > 2025-03-13_00-39-44  | api call to [stage_workflow] with [{'task_key': 'dwfuH8Cz'}]
  | > 2025-03-13_00-39-44  | staging workflow [dwfuH8Cz] with [4] given data instances
  | > 2025-03-13_00-39-45  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
  | > 2025-03-13_00-39-45  | work [/ws/runs/dwfuH8Cz]
  | > 2025-03-13_00-39-45  | data [/msm_home/data]
  | > 2025-03-13_00-39-45  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
  | > 2025-03-13_00-39-45  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/data]
  | > 2025-03-13_00-39-45  | additional params:
  | > 2025-03-13_00-39-45  |     {}
  | > 2025-03-13_00-39-45  | moving remote data libraries to [/msm_home/data]
  | > 2025-03-13_00-39-45  | using nextflow preset [default]
  | > 2025-03-13_00-39

In [8]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-13_00-39-45  | connecting to deployed agent
2025-03-13_00-39-45  | starting relay service


 E| > 2025-03-13_00-39-46 E| relay server already running in [relay/connections]


  | > 2025-03-13_00-39-46  | connecting to relay as [umPrVesV2Rso]
2025-03-13_00-39-47  | executing workflow
  | > including dev binds
  | > 2025-03-13_00-39-47  | api call to [execute_workflow] with [{'key': 'dwfuH8Cz'}]
  | > 2025-03-13_00-39-47  | workspace [/msm_home/runs/dwfuH8Cz]
  | > 2025-03-13_00-39-47  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
  | > 2025-03-13_00-39-47  | executing workflow [dwfuH8Cz] with [2] steps
  | > 2025-03-13_00-39-47  | locating input data with personal globus endpoint
  | > 2025-03-13_00-39-47  | [3RJW32jRo2ju] is at [/msm_home/runs/dwfuH8Cz/_metasmith/task/transforms/3RJW32jRo2ju]
  | > 2025-03-13_00-39-47  | [gzLTT7PL67JN] is at [/msm_home/runs/dwfuH8Cz/_metasmith/task/data/gzLTT7PL67JN]
  | > 2025-03-13_00-39-47  | [zHXmpWcrgYaH] at [/msm_home/data/zHXmpWcrgYaH] is remote [globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb], downloading to [/home/tony/workspace/tools/M

In [9]:
# task = WorkflowTask(
#     plan=plan,
#     agent=agent,
#     data_libraries=[xgdb, refdb],
#     transform_libraries=[trlib],
#     config=dict(
#         nextflow=dict(
#             preset="default",
#             # slurm_account=slurm_account,
#         ),
#     ),
# )